# Testing Different Aspects of Project

In [16]:
import json
from pathlib import Path

import pandas as pd
import requests_cache
from retry_requests import retry
import requests

## Testing API Requests

In [ ]:
# A cached, retrying requests session. Calling the REST endpoint directly
# returns standard JSON (the openmeteo_requests client returns FlatBuffers).
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
session = retry(cache_session, retries=5, backoff_factor=0.2)

In [ ]:
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"start_date": "2026-06-12",
	"end_date": "2026-06-26",
	# Open-Meteo expects a comma-separated list for `daily`
	"daily": ",".join([
		"temperature_2m_mean", "temperature_2m_max", "temperature_2m_min",
		"rain_sum", "snowfall_sum", "wind_speed_10m_max", "daylight_duration",
	]),
	"timezone": "America/Los_Angeles",
	"temperature_unit": "fahrenheit",
	"wind_speed_unit": "mph",
	"precipitation_unit": "inch",
}

resp = requests.get(url, params=params, timeout=30)
resp.raise_for_status()
data = resp.json()
data

{'latitude': 52.54833,
 'longitude': 13.407822,
 'generationtime_ms': 10.132312774658203,
 'utc_offset_seconds': -25200,
 'timezone': 'America/Los_Angeles',
 'timezone_abbreviation': 'GMT-7',
 'elevation': 38.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_mean': '°F',
  'temperature_2m_max': '°F',
  'temperature_2m_min': '°F',
  'rain_sum': 'inch',
  'snowfall_sum': 'inch',
  'wind_speed_10m_max': 'mp/h',
  'daylight_duration': 's'},
 'daily': {'time': ['2026-06-12',
   '2026-06-13',
   '2026-06-14',
   '2026-06-15',
   '2026-06-16',
   '2026-06-17',
   '2026-06-18',
   '2026-06-19',
   '2026-06-20',
   '2026-06-21',
   '2026-06-22',
   '2026-06-23',
   '2026-06-24',
   '2026-06-25',
   '2026-06-26'],
  'temperature_2m_mean': [60.4,
   59.4,
   56.6,
   57.2,
   60.7,
   69.1,
   75.5,
   78.4,
   75.8,
   72.9,
   69.6,
   71.9,
   80.7,
   81.5,
   84.9],
  'temperature_2m_max': [66.0,
   68.4,
   62.6,
   64.8,
   68.9,
   74.5,
   82.4,
   94.1,
   87.4,
   81.5,
   78.7,

In [14]:
# Save the raw JSON response to file (project deliverable)
raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

out_path = raw_dir / f"weather_{params['latitude']}_{params['longitude']}_{params['start_date']}_{params['end_date']}.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(data, f, indent=2)

print(f"Saved raw response to {out_path.resolve()}")

Saved raw response to /Users/williammahnke/Desktop/Revature/weather_project/data/raw/weather_52.52_13.41_2026-06-12_2026-06-26.json


In [15]:
# The "daily" block is already column-oriented, so it loads straight into pandas
daily_df = pd.DataFrame(data["daily"])
daily_df.head()

,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,rain_sum,snowfall_sum,wind_speed_10m_max,daylight_duration
0,2026-06-12,60.4,66.0,56.5,0.181,0.0,9.8,60388.79
1,2026-06-13,59.4,68.4,53.0,0.130,0.0,14.0,60438.52
2,2026-06-14,56.6,62.6,52.4,0.181,0.0,15.9,60482.17
3,2026-06-15,57.2,64.8,49.3,0.024,0.0,15.3,60519.68
4,2026-06-16,60.7,68.9,50.5,0.000,0.0,9.5,60551.06


## Data Processing

In [ ]:
with open("data/raw/san_jose_2025-01-01_2025-12-31.json", "r") as f:
    data = json.load(f)
data